In [1]:
# =========================================
# 1. 패키지
# =========================================
!pip install -q openpyxl scikit-learn

In [2]:
# =========================================
# 2. 파일 업로드
# =========================================
from google.colab import files
uploaded = files.upload()

Saving 모델링_데이터셋_라벨_v3.1.xlsx to 모델링_데이터셋_라벨_v3.1.xlsx


In [5]:
# =========================================
# 3. 데이터 불러오기
# =========================================
import pandas as pd

file_path = "모델링_데이터셋_라벨_v3.1.xlsx"

df = pd.read_excel(
    file_path,
    sheet_name="데이터셋"
)

print(df.shape)
display(df.head())

(2290, 89)


,시군구코드,시도,시군구,기준연도,0~5세 인구,0세 인구,출생아수,어린이집 수,어린이집 정원,어린이집 현원,...,전문의 수(전체),소아청소년과 전문의 수,0~5세 1천명당 소아청소년과 전문의,응급의학과 전문의 수,인구 10만명당 응급의학과 전문의,재정자립도,재정자립도 대체(시도값),0~14세 인구,0~5세 비율,노령화지수
0,11110,서울특별시,종로구,2016,5133,764,846,81,4939,3901,...,1254,65,12.663160,40,26.188808,50.36,0,14494,0.033607,172.140196
1,11140,서울특별시,중구,2016,5085,851,959,67,4136,3446,...,682,19,3.736480,10,7.984096,57.11,0,11527,0.040599,176.871693
2,11170,서울특별시,용산구,2016,9965,1577,1708,125,5363,4830,...,350,31,3.110888,6,2.605965,41.01,0,24369,0.043281,145.869753
3,11200,서울특별시,성동구,2016,14817,2470,2531,189,8133,7049,...,437,30,2.024701,7,2.339111,37.30,0,33892,0.049512,115.484480
4,11215,서울특별시,광진구,2016,15162,2553,2767,217,9297,8046,...,643,41,2.704129,10,2.799435,27.20,0,38537,0.042445,107.148974


In [6]:
target = "공백 위험(최종)"

exclude_cols = [
    "시군구코드",
    "시도",
    "시군구",
    "기준연도",

    # 예측 대상
    "공백 위험(최종)",

    # 다른 라벨
    "어린이집 위험",
    "소아과 위험",
    "고위험(둘 다)",

    # 용도 구분
    "용도",

    # 미래 정보
    "3년 후 어린이집 감소율"
]

In [7]:
feature_cols = [
    col for col in df.columns
    if col not in exclude_cols
    and pd.api.types.is_numeric_dtype(df[col])
]

print("사용 변수 수:", len(feature_cols))

for col in feature_cols:
    print(col)

사용 변수 수: 79
0~5세 인구
0세 인구
출생아수
어린이집 수
어린이집 정원
어린이집 현원
소아과 의원 수
0~5세 1천명당 어린이집
0~5세 1천명당 소아과
정원 충족률
0~5세 대비 정원
출생아/0~5세
0~5세 3년 변화율
출생아 3년 변화율
소아과 3년 변화(곳)
어린이집 1년 변화율
정원 1년 변화율
소아과 0곳(현재 공백)
국공립 수
민간 수
가정 수
직장 수
법인 수
협동 수
국공립 비율
민간 비율
가정 비율
직장 비율
법인 비율
연말 휴지시설 수
연말 휴지시설 비율
1년 폐원 추정 수
1년 신규 추정 수
1년 폐원 추정률
1년 신규 추정률
시도 소아과 개업(t)
시도 소아과 폐업(t)
시도 소아과 순증(t)
시도 소아과 폐업률(t)
시도 소아과 3년 누적 폐업
소아청소년과 진료 병원 수
종합병원 수
병원 수(종합병원·병원)
병원급 의료기관 수
소아과 의원+병원 수
0~5세 1천명당 소아과(의원+병원)
소아과 진료 병원 3년 변화(곳)
소아과 진료 병원 0곳
소아과 의원·병원 모두 0곳
기타 수
기타 비율
1년 휴지 전환 수
운영시설 1년 폐원 추정 수
1년 운영중단 추정 수(폐원+휴지)
1년 운영중단 추정률
당해 휴지 시작 시설 수(휴지시작일 기준)
총인구
65세 이상 인구
20~39세 여성 인구
65세 이상 비율
20~39세 여성 비율
지방소멸위험지수
총인구 1년 변화율
상급종합병원 수(HIRA)
종합병원 수(HIRA)
병원 수(HIRA)
의원 수(HIRA, 전체)
종합병원급 수(상급+종합)
인구 1만명당 의원 수
전문의 수(전체)
소아청소년과 전문의 수
0~5세 1천명당 소아청소년과 전문의
응급의학과 전문의 수
인구 10만명당 응급의학과 전문의
재정자립도
재정자립도 대체(시도값)
0~14세 인구
0~5세 비율
노령화지수


In [8]:
print("\n[미래정보 의심 변수]")

for col in feature_cols:
    if "3년 후" in col or "라벨" in col:
        print(col)


[미래정보 의심 변수]


In [9]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

rf = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=3,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    )
])

In [10]:
train_df = df[
    (df["기준연도"] >= 2016) &
    (df["기준연도"] <= 2022)
].copy()

X_train = train_df[feature_cols]
y_train = train_df[target]

print("학습 행 수:", len(train_df))
print("양성 개수:", y_train.sum())
print("양성 비율:", y_train.mean())

학습 행 수: 1603
양성 개수: 418.0
양성 비율: 0.2607610729881472


In [11]:
rf.fit(X_train, y_train)

print("랜덤포레스트 학습 완료")

랜덤포레스트 학습 완료


In [12]:
df_2024 = df[
    df["기준연도"] == 2024
].copy()

X_2024 = df_2024[feature_cols]

risk_2027 = rf.predict_proba(X_2024)[:, 1]

In [13]:
result_2027 = df_2024[
    ["시군구코드", "시도", "시군구"]
].copy()

result_2027["예측연도"] = 2027
result_2027["위험점수"] = risk_2027

result_2027["위험순위"] = (
    result_2027["위험점수"]
    .rank(method="min", ascending=False)
    .astype(int)
)

result_2027 = result_2027.sort_values(
    "위험점수",
    ascending=False
).reset_index(drop=True)

display(result_2027.head(20))

,시군구코드,시도,시군구,예측연도,위험점수,위험순위
0,48310,경상남도,거제시,2027,0.796344,1
1,48250,경상남도,김해시,2027,0.770351,2
2,44130,충청남도,천안시,2027,0.744203,3
3,48120,경상남도,창원시,2027,0.719408,4
4,41250,경기도,동두천시,2027,0.660227,5
5,48330,경상남도,양산시,2027,0.652985,6
6,26710,부산광역시,기장군,2027,0.626572,7
7,28245,인천광역시,계양구,2027,0.609877,8
8,41410,경기도,군포시,2027,0.609630,9
9,48730,경상남도,함안군,2027,0.608114,10


In [14]:
df_2025 = df[
    df["기준연도"] == 2025
].copy()

X_2025 = df_2025[feature_cols]

risk_2028 = rf.predict_proba(X_2025)[:, 1]

In [15]:
result_2028 = df_2025[
    ["시군구코드", "시도", "시군구"]
].copy()

result_2028["예측연도"] = 2028
result_2028["위험점수"] = risk_2028

result_2028["위험순위"] = (
    result_2028["위험점수"]
    .rank(method="min", ascending=False)
    .astype(int)
)

result_2028 = result_2028.sort_values(
    "위험점수",
    ascending=False
).reset_index(drop=True)

display(result_2028.head(20))

,시군구코드,시도,시군구,예측연도,위험점수,위험순위
0,48310,경상남도,거제시,2028,0.714422,1
1,26530,부산광역시,사상구,2028,0.648653,2
2,46110,전라남도,목포시,2028,0.635919,3
3,41250,경기도,동두천시,2028,0.603678,4
4,26710,부산광역시,기장군,2028,0.585485,5
5,28245,인천광역시,계양구,2028,0.569797,6
6,26380,부산광역시,사하구,2028,0.564756,7
7,29200,광주광역시,광산구,2028,0.554873,8
8,30230,대전광역시,대덕구,2028,0.545088,9
9,48250,경상남도,김해시,2028,0.534575,10


In [16]:
trend_df = result_2027[
    [
        "시군구코드",
        "시도",
        "시군구",
        "위험점수",
        "위험순위"
    ]
].copy()

trend_df = trend_df.rename(
    columns={
        "위험점수": "2027_위험점수",
        "위험순위": "2027_위험순위"
    }
)

In [17]:
temp_2028 = result_2028[
    [
        "시군구코드",
        "위험점수",
        "위험순위"
    ]
].copy()

temp_2028 = temp_2028.rename(
    columns={
        "위험점수": "2028_위험점수",
        "위험순위": "2028_위험순위"
    }
)

trend_df = trend_df.merge(
    temp_2028,
    on="시군구코드",
    how="left"
)

In [18]:
trend_df["위험점수_변화"] = (
    trend_df["2028_위험점수"]
    - trend_df["2027_위험점수"]
)

In [19]:
trend_df = trend_df.sort_values(
    "위험점수_변화",
    ascending=False
).reset_index(drop=True)

display(trend_df.head(20))

,시군구코드,시도,시군구,2027_위험점수,2027_위험순위,2028_위험점수,2028_위험순위,위험점수_변화
0,41570,경기도,김포시,0.181348,153,0.343945,53,0.162597
1,52140,전북특별자치도,익산시,0.240395,120,0.343760,54,0.103365
2,30110,대전광역시,동구,0.370946,48,0.465624,20,0.094678
3,41550,경기도,안성시,0.233903,127,0.313650,67,0.079746
4,46130,전라남도,여수시,0.157907,166,0.232867,116,0.074960
5,51210,강원특별자치도,속초시,0.296575,91,0.371505,40,0.074929
6,41310,경기도,구리시,0.294397,93,0.367049,41,0.072653
7,41670,경기도,여주시,0.388956,44,0.460749,23,0.071794
8,29110,광주광역시,동구,0.108941,200,0.176225,155,0.067284
9,46810,전라남도,강진군,0.079963,218,0.147007,181,0.067043


In [20]:
top20_2027_codes = set(
    result_2027.head(20)["시군구코드"]
)

top20_2028_codes = set(
    result_2028.head(20)["시군구코드"]
)

common_top20 = (
    top20_2027_codes &
    top20_2028_codes
)

trend_df["2년연속_상위20"] = (
    trend_df["시군구코드"]
    .isin(common_top20)
    .astype(int)
)

In [21]:
display(
    trend_df[
        trend_df["2년연속_상위20"] == 1
    ]
)

,시군구코드,시도,시군구,2027_위험점수,2027_위험순위,2028_위험점수,2028_위험순위,위험점수_변화,2년연속_상위20
12,46110,전라남도,목포시,0.569984,15,0.635919,3,0.065935,1
16,26530,부산광역시,사상구,0.590566,12,0.648653,2,0.058087,1
84,29200,광주광역시,광산구,0.547686,19,0.554873,8,0.007187,1
172,28245,인천광역시,계양구,0.609877,8,0.569797,6,-0.040080,1
173,26710,부산광역시,기장군,0.626572,7,0.585485,5,-0.041087,1
182,41250,경기도,동두천시,0.660227,5,0.603678,4,-0.056548,1
186,48820,경상남도,고성군,0.550541,18,0.486155,17,-0.064386,1
188,48220,경상남도,통영시,0.598661,11,0.525320,11,-0.073341,1
194,48310,경상남도,거제시,0.796344,1,0.714422,1,-0.081922,1
196,41410,경기도,군포시,0.609630,9,0.523664,12,-0.085966,1


In [22]:
new_high_2028 = (
    top20_2028_codes
    - top20_2027_codes
)

trend_df["2028_신규상위20"] = (
    trend_df["시군구코드"]
    .isin(new_high_2028)
    .astype(int)
)

In [23]:
rf_model = rf.named_steps["model"]

importance_df = pd.DataFrame({
    "변수": feature_cols,
    "중요도": rf_model.feature_importances_
})

importance_df = importance_df.sort_values(
    "중요도",
    ascending=False
).reset_index(drop=True)

display(importance_df.head(20))

,변수,중요도
0,0~5세 1천명당 어린이집,0.066053
1,기타 비율,0.033318
2,0~5세 3년 변화율,0.026575
3,가정 비율,0.025944
4,민간 비율,0.024405
5,정원 충족률,0.024186
6,총인구 1년 변화율,0.023779
7,법인 비율,0.023548
8,출생아 3년 변화율,0.022743
9,시도 소아과 3년 누적 폐업,0.021984


In [24]:
output_file = "2027_2028_공백위험_예측결과_RF.xlsx"

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    result_2027.to_excel(
        writer,
        sheet_name="2027_예측",
        index=False
    )

    result_2028.to_excel(
        writer,
        sheet_name="2028_예측",
        index=False
    )

    result_2027.head(20).to_excel(
        writer,
        sheet_name="2027_상위20",
        index=False
    )

    result_2028.head(20).to_excel(
        writer,
        sheet_name="2028_상위20",
        index=False
    )

    trend_df.to_excel(
        writer,
        sheet_name="위험도_변화",
        index=False
    )

    importance_df.to_excel(
        writer,
        sheet_name="변수중요도",
        index=False
    )

In [25]:
from google.colab import files

files.download(
    "2027_2028_공백위험_예측결과_RF.xlsx"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>